In [ ]:
"""
AlphaFold3:
python ipsae.py <path_to_af3_json_file> <path_to_af3_cif_file> <pae_cutoff> <dist_cutoff>                    
python ipsae.py fold_aurka_tpx2_full_data_0.json fold_aurka_tpx2_model_0.cif 10 10
"""


import os
import sys

IPSAE_PATH = "/home/junjiechen/1_work/original_soft/IPSAE/ipsae.py"
dirs = ["msa_notemplate-new", "msa_template", "nomsa_template", "nomsa_notemplate"]

with open("job.list", "w") as f:
    counts = {}
    for dir in sorted(dirs):
        dir_path = os.path.join("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/alphafold3/predict/original_result", dir)
        cnt = 0
        for root, _, files in sorted(os.walk(dir_path)):
            for file in sorted(files):
                if file.endswith(".cif") and "seed-" in file and "sample-" in file:
                    cif_path = os.path.join(root, file)
                    json_path = cif_path.replace("model.cif", "confidences.json")
                    f.write(f"python {IPSAE_PATH} {json_path} {cif_path} 10 10\n")
                    cnt += 1
        counts[dir] = cnt
        print(f"{dir}: {cnt} lines")

total = sum(counts.values())
expected = 15 * 170 * 4
print(f"\nTotal: {total} lines (expected {expected})")
assert total == expected, f"MISMATCH: expected {expected}, got {total}"
print("All OK!")


msa_notemplate-new: 2550 lines
msa_template: 2550 lines
nomsa_notemplate: 2550 lines
nomsa_template: 2550 lines

Total: 10200 lines (expected 10200)
All OK!


### 后续所有的msa_notemplate-new删去-new后缀，表示与剩下的三组在相同的条件下进行结构预测

### 原有的msa_notemplate增加-old后缀

In [2]:
import os
import pandas as pd

dirs = ["msa_notemplate", "msa_template", "nomsa_template", "nomsa_notemplate"]

for dir in sorted(dirs):
    with open(f"{dir}_ipsae.csv", "w") as f:
        f.write("complex,seed,id,ipSAE_max,ipSAE_min\n")
        dir_path = os.path.join("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/alphafold3/predict/original_result", dir)
        for root, _, files in sorted(os.walk(dir_path)):
            for file in sorted(files):
                if file.endswith("_10_10.txt") and "seed-" in file and "sample-" in file:
                    file_path = os.path.join(root, file)
                    complex_name = file.split("_")[0]
                    seed = file.split("seed-")[1].split("_")[0]
                    sample_id = file.split("sample-")[1].split("_")[0]
                    df = pd.read_csv(file_path, sep=r"\s+")
                    ipsae_max = df["ipSAE"].max()
                    ipsae_min = df["ipSAE"].min()
                    f.write(f"{complex_name},{seed},{sample_id},{ipsae_max},{ipsae_min}\n")